# Synthetic Control: Measuring 'Shop the Look' Feature Impact
*E-commerce · Marketing Science · Causal Inference*

**Olivia Pan** — [github.com/olieepop](https://github.com/olieepop)

---

## The question

A global e-commerce company launches 'Shop the Look' — a visual commerce feature that lets users shop a complete outfit or product bundle directly from an inspirational image — exclusively in North America.

The business wants to know three things:

1. **Did it move conversion?** (primary)
2. **Did it expand basket size, or just shift existing behavior?** (substitution vs. expansion)
3. **Did it create downstream engagement beyond the transaction?** (halo effect)

Simple question. Hard measurement problem.

---

## Why this is harder than it looks

The instinct is to run a standard pre/post comparison or a Difference-in-Differences model. Both break down here for the same reason:

**'Shop the Look' is a UI feature — it can't be withheld from specific users within a market.** Once it's live in North America, every NA user sees it. There's no clean user-level control group.

The natural alternative is to use other regions (EMEA, APLA) as controls — but that only works if they were on parallel trajectories with NA before the launch. If NA was already outperforming or underperforming other regions, any post-launch difference is contaminated by that pre-existing gap.

We'll show that parallel trends fails here. Then we'll show what to do about it.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BLUE   = '#111111'
RED    = '#C8102E'
GRAY   = '#8C8C8C'
LGRAY  = '#E5E5E5'

np.random.seed(2024)
print('ready.')

---

## Step 1 — Show why DiD breaks down here

Before reaching for Synthetic Control, we need to demonstrate the problem it's solving. The parallel trends assumption — the foundation of DiD — requires that the treated unit and control units were moving together before the intervention. If they weren't, DiD will attribute pre-existing divergence to the treatment.

North America in e-commerce is structurally different from EMEA and APLA: higher baseline conversion, different seasonality driven by holiday shopping cycles, and a more mature mobile commerce behavior. These aren't random — they're structural, persistent differences that make a simple average of other regions a poor counterfactual for NA.

We'll build the data first, then show the failed parallel trends check explicitly.

In [ ]:
# ── Timeline ───────────────────────────────────────────────────
PRE_WEEKS  = 26   # 6 months pre-launch — SC needs a long pre-period to fit well
POST_WEEKS = 12   # 3 months post-launch
TOTAL_WEEKS = PRE_WEEKS + POST_WEEKS
weeks = np.arange(1, TOTAL_WEEKS + 1)
launch_week = PRE_WEEKS + 1

# ── Donor countries by region ──────────────────────────────────
# NA is the treated unit — everything else is the donor pool
donors = {
    # EMEA
    'UK':          {'base': 0.042, 'trend': 0.0003,  'seasonal_amp': 0.004, 'noise': 0.0012},
    'Germany':     {'base': 0.038, 'trend': 0.00025, 'seasonal_amp': 0.003, 'noise': 0.0010},
    'France':      {'base': 0.035, 'trend': 0.00020, 'seasonal_amp': 0.003, 'noise': 0.0011},
    'Netherlands': {'base': 0.033, 'trend': 0.00018, 'seasonal_amp': 0.002, 'noise': 0.0009},
    # APLA
    'Australia':   {'base': 0.040, 'trend': 0.00022, 'seasonal_amp': 0.005, 'noise': 0.0013},
    'Japan':       {'base': 0.036, 'trend': 0.00015, 'seasonal_amp': 0.002, 'noise': 0.0008},
    'South Korea': {'base': 0.034, 'trend': 0.00028, 'seasonal_amp': 0.003, 'noise': 0.0010},
    'Brazil':      {'base': 0.028, 'trend': 0.00030, 'seasonal_amp': 0.006, 'noise': 0.0015},
    'Mexico':      {'base': 0.026, 'trend': 0.00025, 'seasonal_amp': 0.005, 'noise': 0.0014},
}

# NA has structurally higher conversion + different seasonal shape
# This is what breaks parallel trends — NA isn't just a scaled version of other markets
na_params = {'base': 0.058, 'trend': 0.00035, 'seasonal_amp': 0.008, 'noise': 0.0014}
TRUE_LIFT = 0.022  # ~2.2pp absolute lift from Shop the Look — realistic for a UX feature

def generate_series(params, weeks, treatment_effect=0, launch_week=None):
    """Generate weekly conversion rate series with trend, seasonality, and noise."""
    base        = params['base']
    trend       = params['trend']
    amp         = params['seasonal_amp']
    noise_sd    = params['noise']
    series = []
    for w in weeks:
        seasonal = amp * np.sin(2 * np.pi * w / 52)
        lift     = treatment_effect if (launch_week and w >= launch_week) else 0
        val      = base + trend * w + seasonal + lift + np.random.normal(0, noise_sd)
        series.append(max(val, 0))
    return np.array(series)

# Generate all series
na_series     = generate_series(na_params, weeks, TRUE_LIFT, launch_week)
donor_series  = {name: generate_series(p, weeks) for name, p in donors.items()}

# Build dataframe
df = pd.DataFrame({'week': weeks, 'NA': na_series, **donor_series})
df['post'] = (df.week >= launch_week).astype(int)

print(f'Markets: NA (treated) + {len(donors)} donor countries')
print(f'Timeline: {PRE_WEEKS}wk pre-launch + {POST_WEEKS}wk post-launch')
print(f'Injected lift: {TRUE_LIFT:.1%} absolute — DiD should recover this')
print(f"\nNA pre-period avg conversion: {df[df.post==0]['NA'].mean():.3%}")
print(f"Donor pre-period avg conversion: {df[df.post==0][list(donors.keys())].mean().mean():.3%}")

### The parallel trends problem — made visible

In [ ]:
pre_df       = df[df.post == 0].copy()
donor_cols   = list(donors.keys())
simple_avg   = pre_df[donor_cols].mean(axis=1)

# Correlation between NA and simple average of donors in pre-period
corr, pval = stats.pearsonr(pre_df['NA'].values, simple_avg.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw pre-period trajectories
ax = axes[0]
for country in donor_cols:
    ax.plot(pre_df.week, pre_df[country], color=LGRAY, lw=1.2, alpha=0.8)
ax.plot(pre_df.week, simple_avg, color=GRAY, lw=2, linestyle='--', label='Donor average')
ax.plot(pre_df.week, pre_df['NA'], color=RED, lw=2.5, label='North America')
ax.set_title('Pre-Period Trajectories', fontsize=12, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Conversion rate')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.legend(fontsize=9)
ax.text(0.05, 0.92, 'NA sits structurally above donor avg\n→ parallel trends assumption fails',
        transform=ax.transAxes, fontsize=8.5, color='#555',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# Right: level difference over time
ax2 = axes[1]
gap = pre_df['NA'].values - simple_avg.values
ax2.bar(pre_df.week, gap, color=[RED if g > 0 else GRAY for g in gap], alpha=0.6)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(gap.mean(), color=RED, lw=1.5, linestyle='--',
            label=f'Mean gap: {gap.mean():.2%}')
ax2.set_title(f'NA vs. Donor Average Gap (Pre-Period)\nCorrelation: {corr:.3f}', 
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Week')
ax2.set_ylabel('Conversion rate gap')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2%}'))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('parallel_trends_failure.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pre-period correlation (NA vs. donor avg): {corr:.3f}')
print(f'Mean level gap: {gap.mean():.3%}')
print()
print('DiD verdict: parallel trends fails. A simple donor average is a bad counterfactual for NA.')
print('The gap is persistent and structural — not noise. We need a better synthetic control.')

---

## Step 2 — Build the Synthetic Control

Instead of averaging donors equally, Synthetic Control finds the *optimal weighted combination* of donor units that best reproduces NA's pre-period conversion trajectory. The weights are solved via constrained optimization — they must be non-negative and sum to 1 (no extrapolation, no leverage effects).

**The objective:** minimize the mean squared prediction error (MSPE) between NA and the weighted donor combination during the pre-period.

**Why this matters:** a well-fit synthetic unit is a credible counterfactual precisely because it wasn't constructed by averaging — it was constructed by finding the donor mix that would have looked like NA if NA hadn't been treated. That's a fundamentally stronger claim than "here's the average of other markets."

Once the weights are set on pre-period data, they're frozen. We never touch them again. The post-period gap between NA and Synthetic NA is the treatment effect estimate — no further tuning allowed.

In [ ]:
pre_na     = df[df.post == 0]['NA'].values
pre_donors = df[df.post == 0][donor_cols].values  # shape: (PRE_WEEKS, n_donors)

def mspe(weights):
    """Mean squared prediction error between NA and weighted donor combination."""
    synthetic = pre_donors @ weights
    return np.mean((pre_na - synthetic) ** 2)

n_donors = len(donor_cols)
w0       = np.ones(n_donors) / n_donors  # start from equal weights

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # weights sum to 1
bounds      = [(0, 1)] * n_donors                              # non-negative weights

result  = minimize(mspe, w0, method='SLSQP', bounds=bounds, constraints=constraints,
                   options={'ftol': 1e-12, 'maxiter': 2000})
weights = result.x

# Apply frozen weights to full timeline
all_donors     = df[donor_cols].values
synthetic_na   = all_donors @ weights
df['synthetic'] = synthetic_na

# Pre-period fit quality
pre_mspe  = np.mean((pre_na - synthetic_na[:PRE_WEEKS]) ** 2)
pre_corr  = np.corrcoef(pre_na, synthetic_na[:PRE_WEEKS])[0, 1]
naive_mspe = np.mean((pre_na - pre_donors.mean(axis=1)) ** 2)

print('Synthetic control weights:')
print('-' * 35)
for country, w in sorted(zip(donor_cols, weights), key=lambda x: -x[1]):
    bar = '█' * int(w * 40)
    print(f'  {country:<14} {w:.3f}  {bar}')
print()
print(f'Pre-period fit')
print(f'  MSPE (synthetic):    {pre_mspe:.8f}')
print(f'  MSPE (naive avg):    {naive_mspe:.8f}')
print(f'  Improvement:         {(1 - pre_mspe/naive_mspe):.1%} better fit than equal-weight average')
print(f'  Pre-period corr:     {pre_corr:.4f}')

### Validating the synthetic fit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full timeline
ax = axes[0]
ax.axvspan(launch_week - 0.5, TOTAL_WEEKS + 0.5, alpha=0.06, color=RED)
ax.axvline(launch_week - 0.5, color=GRAY, linestyle='--', lw=1)
ax.plot(df.week, df['NA'],        color=RED,  lw=2.5, label='North America (observed)')
ax.plot(df.week, df['synthetic'], color=BLUE, lw=2,   linestyle='--', label='Synthetic NA (counterfactual)')
ax.set_title("North America vs. Synthetic NA", fontsize=12, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Conversion rate')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.text(launch_week + 0.5, df['NA'].min() + 0.001, "'Shop the Look'\nlaunched",
        fontsize=8, color=GRAY)
ax.legend(fontsize=9)

# Right: pre-period residuals
ax2 = axes[1]
residuals = pre_na - synthetic_na[:PRE_WEEKS]
ax2.bar(range(1, PRE_WEEKS + 1), residuals,
        color=[RED if r > 0 else BLUE for r in residuals], alpha=0.6)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(residuals.mean(), color=GRAY, lw=1.5, linestyle='--',
            label=f'Mean residual: {residuals.mean():.4%}')
ax2.set_title('Pre-Period Fit: Residuals (NA − Synthetic NA)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Pre-period week')
ax2.set_ylabel('Residual (conversion rate)')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.3%}'))
ax2.legend(fontsize=9)

fit_status = '✅ good fit — residuals small and unbiased' if abs(residuals.mean()) < 0.001 else '⚠️ review fit before proceeding'
ax2.text(0.05, 0.92, fit_status, transform=ax2.transAxes, fontsize=9,
         color='green' if '✅' in fit_status else 'darkorange',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.savefig('synthetic_control_fit.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Step 3 — Primary outcome: Conversion rate

With a well-fit synthetic control, the post-period gap between NA and Synthetic NA is our treatment effect estimate. This is the conversion lift attributable to 'Shop the Look.'

The gap is clean to read: whatever Synthetic NA does post-launch represents what NA *would have done* without the feature. The difference is caused by the feature — not by seasonality, not by macro trends, not by pre-existing level differences. Those are all already absorbed into the synthetic weights.

In [ ]:
post_df       = df[df.post == 1].copy()
na_post       = post_df['NA'].values
synthetic_post = post_df['synthetic'].values
gaps          = na_post - synthetic_post

avg_gap       = gaps.mean()
pre_baseline  = df[df.post == 0]['synthetic'].mean()
relative_lift = avg_gap / pre_baseline

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gap chart
ax = axes[0]
ax.axvspan(launch_week - 0.5, TOTAL_WEEKS + 0.5, alpha=0.06, color=RED)
ax.axvline(launch_week - 0.5, color=GRAY, linestyle='--', lw=1)
ax.plot(df.week, df['NA'],        color=RED,  lw=2.5, label='NA (observed)')
ax.plot(df.week, df['synthetic'], color=BLUE, lw=2,   linestyle='--', label='Synthetic NA')
ax.fill_between(post_df.week, synthetic_post, na_post,
                alpha=0.15, color=RED, label=f'Causal gap (~{avg_gap:.2%} abs)')
ax.set_title('Conversion Rate: Observed vs. Counterfactual', fontsize=12, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Conversion rate')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.legend(fontsize=9)

# Right: weekly gap bars
ax2 = axes[1]
bars = ax2.bar(post_df.week, gaps * 100,
               color=[RED if g > 0 else GRAY for g in gaps], alpha=0.75)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(avg_gap * 100, color=RED, lw=2, linestyle='--',
            label=f'Avg gap: {avg_gap:.2%}')
ax2.set_title('Weekly Causal Gap (NA − Synthetic NA)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Post-launch week')
ax2.set_ylabel('Gap (percentage points)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('conversion_lift.png', dpi=150, bbox_inches='tight')
plt.show()

print('Primary outcome: Conversion rate')
print('-' * 45)
print(f'Avg post-launch gap:    {avg_gap:.3%} absolute')
print(f'Relative lift:          {relative_lift:.1%}')
print(f'True injected lift:     {TRUE_LIFT:.3%}')
print(f'Recovery error:         {abs(avg_gap - TRUE_LIFT)/TRUE_LIFT:.1%}')

---

## Step 4 — Significance testing: Placebo permutation tests

Standard p-values require enough observations to invoke the central limit theorem. With 9 donor units, that assumption doesn't hold — we can't run a t-test and report a p-value with a straight face.

Synthetic Control handles this differently: **placebo tests**.

The logic: if 'Shop the Look' truly caused the NA gap, then running the exact same SC procedure on donor units — which received *no* treatment — should produce gaps that are much smaller. We run SC for every donor country as if it were the treated unit, collect all their post-period gaps, and ask: how unusual is NA's gap relative to that distribution?

This is essentially a permutation test. NA's result is significant if its gap is larger than what we'd expect by chance — where 'chance' is defined by the gaps we observe in untreated units.

**One refinement:** donors with poor pre-period fit make noisy placebos. We filter out any donor whose pre-period MSPE is more than 2x NA's — a standard practice in the SC literature. Bad-fitting placebos inflate the null distribution and make it harder to detect real effects.

In [ ]:
def run_sc(treated_series, donor_matrix, pre_weeks):
    """Run synthetic control for a given treated unit. Returns weights, synthetic series, pre-MSPE."""
    pre_treated = treated_series[:pre_weeks]
    pre_donors  = donor_matrix[:pre_weeks]
    n           = donor_matrix.shape[1]

    result = minimize(
        lambda w: np.mean((pre_treated - pre_donors @ w) ** 2),
        np.ones(n) / n,
        method='SLSQP',
        bounds=[(0, 1)] * n,
        constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        options={'ftol': 1e-12, 'maxiter': 2000}
    )
    w         = result.x
    synthetic = donor_matrix @ w
    pre_mspe  = np.mean((pre_treated - synthetic[:pre_weeks]) ** 2)
    return w, synthetic, pre_mspe

# NA's pre-period MSPE — our benchmark for filtering placebos
na_pre_mspe = np.mean((df[df.post==0]['NA'].values - synthetic_na[:PRE_WEEKS]) ** 2)

# Run placebo SC for each donor country
placebo_gaps = {}
placebo_mspe = {}

for i, country in enumerate(donor_cols):
    # Donor pool for this placebo = all other donors (excluding the one being 'treated')
    placebo_donor_cols = [c for c in donor_cols if c != country]
    placebo_matrix     = df[placebo_donor_cols].values
    treated_series     = df[country].values

    _, synthetic_placebo, p_mspe = run_sc(treated_series, placebo_matrix, PRE_WEEKS)
    post_gap = (treated_series[PRE_WEEKS:] - synthetic_placebo[PRE_WEEKS:]).mean()

    placebo_gaps[country] = post_gap
    placebo_mspe[country] = p_mspe

# Filter: keep only placebos with pre-MSPE ≤ 2× NA's MSPE
mspe_threshold     = 2 * na_pre_mspe
valid_placebos     = {c: g for c, g in placebo_gaps.items() if placebo_mspe[c] <= mspe_threshold}
filtered_placebos  = {c: g for c, g in placebo_gaps.items() if placebo_mspe[c] > mspe_threshold}

all_gaps_valid = list(valid_placebos.values())
na_rank        = sum(abs(avg_gap) > abs(g) for g in all_gaps_valid)
p_value_approx = 1 - na_rank / (len(all_gaps_valid) + 1)

print(f'NA pre-period MSPE:      {na_pre_mspe:.2e}')
print(f'MSPE filter threshold:   {mspe_threshold:.2e}  (2× NA MSPE)')
print(f'Valid placebos:          {len(valid_placebos)} of {len(donor_cols)}')
if filtered_placebos:
    print(f'Filtered out (poor fit): {", ".join(filtered_placebos.keys())}')
print()
print(f'NA post-period gap:      {avg_gap:.4%}')
print(f'Placebo gaps (valid):    {[f"{g:.4%}" for g in all_gaps_valid]}')
print(f'NA rank (by abs gap):    {na_rank} of {len(all_gaps_valid)}')
print(f'Approx. p-value:         {p_value_approx:.3f}')

In [ ]:
# Visualize placebo distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gap distribution
ax = axes[0]
placebo_vals = sorted(all_gaps_valid)
ax.barh(range(len(placebo_vals)), [g * 100 for g in placebo_vals],
        color=LGRAY, edgecolor=GRAY, alpha=0.8, label='Placebo gaps (donor countries)')
ax.axvline(avg_gap * 100, color=RED, lw=2.5, linestyle='-',
           label=f'NA gap: {avg_gap:.2%}')
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks([])
ax.set_title('Placebo Distribution vs. NA Gap', fontsize=12, fontweight='bold')
ax.set_xlabel('Post-period gap (percentage points)')
ax.legend(fontsize=9)
sig_text = f'p ≈ {p_value_approx:.2f}\n{"Significant" if p_value_approx < 0.1 else "Not significant"} at 10% level'
ax.text(0.65, 0.12, sig_text, transform=ax.transAxes, fontsize=9,
        color=RED if p_value_approx < 0.1 else GRAY,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# Right: run chart — NA vs all placebos post-period
ax2 = axes[1]
post_weeks_range = post_df.week.values
for country in valid_placebos:
    placebo_donor_cols = [c for c in donor_cols if c != country]
    placebo_matrix     = df[placebo_donor_cols].values
    _, synth_p, _      = run_sc(df[country].values, placebo_matrix, PRE_WEEKS)
    placebo_series     = df[country].values[PRE_WEEKS:] - synth_p[PRE_WEEKS:]
    ax2.plot(post_weeks_range, placebo_series * 100, color=LGRAY, lw=1, alpha=0.7)

na_gap_series = na_post - synthetic_post
ax2.plot(post_weeks_range, na_gap_series * 100, color=RED, lw=2.5, label='North America')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Post-Period Gaps: NA vs. Placebos', fontsize=12, fontweight='bold')
ax2.set_xlabel('Week')
ax2.set_ylabel('Gap (percentage points)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('placebo_tests.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Step 5 — Secondary effects

Conversion rate is the headline. But it doesn't tell the full story of what 'Shop the Look' actually changed.

Three secondary questions matter for the investment decision:

- **Basket expansion:** did users add more items per order, or did conversion lift come from the same basket size? If AOV didn't move, the feature may have just accelerated decisions rather than changed them.
- **Substitution:** did 'Shop the Look' pull revenue from existing organic browse paths, or grow the total pie? A feature that cannibalizes its own category isn't a win.
- **Halo effect:** did users who engaged with the feature come back more? Session depth and repeat visit rate signal whether this built habit or just captured intent.

The elegant part: we use the same synthetic weights to estimate all three. Build the counterfactual once, apply it to every outcome. This keeps the causal logic consistent and avoids the multiple-testing problem of running separate models per metric.

In [ ]:
# Generate secondary outcome series for NA and donors
# Each gets its own true effect — realistic effect sizes for a UX feature

secondary_outcomes = {
    'Avg Order Value ($)': {
        'na_base': 95.0, 'donor_base': 82.0, 'trend': 0.08,
        'seasonal_amp': 3.0, 'noise': 2.5, 'true_lift': 8.5,  # $8.50 AOV lift
        'unit': '$', 'interpretation': 'basket expansion'
    },
    'Bundle Mix Rate (%)': {
        'na_base': 0.31, 'donor_base': 0.27, 'trend': 0.0002,
        'seasonal_amp': 0.01, 'noise': 0.008, 'true_lift': 0.045,  # +4.5pp bundle rate
        'unit': '%', 'interpretation': 'substitution/expansion'
    },
    'Repeat Visit Rate (%)': {
        'na_base': 0.22, 'donor_base': 0.19, 'trend': 0.00008,
        'seasonal_amp': 0.005, 'noise': 0.006, 'true_lift': 0.018,  # +1.8pp repeat rate
        'unit': '%', 'interpretation': 'halo effect'
    }
}

results_summary = []

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (metric, params) in enumerate(secondary_outcomes.items()):
    ax = axes[idx]

    # Generate NA series for this metric
    na_metric = []
    for w in weeks:
        seasonal = params['seasonal_amp'] * np.sin(2 * np.pi * w / 52)
        lift     = params['true_lift'] if w >= launch_week else 0
        val      = params['na_base'] + params['trend'] * w + seasonal + lift + np.random.normal(0, params['noise'])
        na_metric.append(val)
    na_metric = np.array(na_metric)

    # Generate donor series for this metric
    donor_metric = np.zeros((TOTAL_WEEKS, len(donor_cols)))
    for j, country in enumerate(donor_cols):
        for w_idx, w in enumerate(weeks):
            seasonal = params['seasonal_amp'] * 0.8 * np.sin(2 * np.pi * w / 52)
            val      = params['donor_base'] + params['trend'] * w + seasonal + np.random.normal(0, params['noise'])
            donor_metric[w_idx, j] = val

    # Apply frozen SC weights — same weights, new outcome
    synthetic_metric = donor_metric @ weights

    post_na_metric        = na_metric[PRE_WEEKS:]
    post_synthetic_metric = synthetic_metric[PRE_WEEKS:]
    metric_gap            = (post_na_metric - post_synthetic_metric).mean()
    pre_baseline_metric   = synthetic_metric[:PRE_WEEKS].mean()
    relative_metric_lift  = metric_gap / pre_baseline_metric

    results_summary.append({
        'Metric'         : metric,
        'Effect'         : params['interpretation'],
        'Avg gap'        : metric_gap,
        'Relative lift'  : relative_metric_lift,
        'True lift'      : params['true_lift'],
        'Recovery error' : abs(metric_gap - params['true_lift']) / abs(params['true_lift'])
    })

    # Plot
    ax.axvspan(launch_week - 0.5, TOTAL_WEEKS + 0.5, alpha=0.06, color=RED)
    ax.axvline(launch_week - 0.5, color=GRAY, linestyle='--', lw=1)
    ax.plot(weeks, na_metric,        color=RED,  lw=2, label='NA (observed)')
    ax.plot(weeks, synthetic_metric, color=BLUE, lw=1.8, linestyle='--', label='Synthetic NA')
    ax.fill_between(weeks[PRE_WEEKS:], post_synthetic_metric, post_na_metric,
                    alpha=0.15, color=RED)
    ax.set_title(f'{metric}\n({params["interpretation"]})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Week')
    if params['unit'] == '%':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
    else:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}'))
    if idx == 0:
        ax.legend(fontsize=8)
    ax.text(0.05, 0.92, f'Gap: {metric_gap:+.3f}\n({relative_metric_lift:+.1%})',
            transform=ax.transAxes, fontsize=9, color=RED,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.suptitle("Secondary Effects: Same SC Weights, Multiple Outcomes", 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('secondary_effects.png', dpi=150, bbox_inches='tight')
plt.show()

results_df = pd.DataFrame(results_summary)
print(results_df[['Metric', 'Effect', 'Avg gap', 'Relative lift', 'Recovery error']].to_string(index=False))

---

## Step 6 — Business recommendation

Four metrics, four SC estimates, one decision.

The question isn't just whether 'Shop the Look' moved conversion. It's whether the *pattern of effects* tells a coherent story about what it actually did — and whether that story justifies the investment to scale globally.

In [ ]:
# Business inputs
NA_MONTHLY_VISITORS  = 4_000_000
PRE_CONVERSION       = pre_baseline
POST_CONVERSION      = PRE_CONVERSION + avg_gap
AVG_ORDER_VALUE_PRE  = 95.0
AOV_LIFT             = results_summary[0]['Avg gap']  # from SC estimate
AVG_ORDER_VALUE_POST = AVG_ORDER_VALUE_PRE + AOV_LIFT

# Revenue impact
rev_pre  = NA_MONTHLY_VISITORS * PRE_CONVERSION  * AVG_ORDER_VALUE_PRE
rev_post = NA_MONTHLY_VISITORS * POST_CONVERSION * AVG_ORDER_VALUE_POST
rev_lift = rev_post - rev_pre
rev_lift_pct = rev_lift / rev_pre

bundle_lift  = results_summary[1]['Avg gap']
repeat_lift  = results_summary[2]['Avg gap']

print('SHOP THE LOOK — NORTH AMERICA RESULTS')
print('=' * 60)
print()
print('Primary outcome')
print(f'  Conversion rate lift:    {avg_gap:.2%} absolute  ({relative_lift:.1%} relative)')
print(f'  Approx. significance:    p ≈ {p_value_approx:.2f}')
print()
print('Secondary effects')
print(f'  Avg order value lift:    ${AOV_LIFT:.2f}  → basket expansion confirmed')
print(f'  Bundle mix rate lift:    {bundle_lift:.2%}  → additive, not substitution')
print(f'  Repeat visit rate lift:  {repeat_lift:.2%}  → halo effect present')
print()
print('Revenue translation (NA, monthly)')
print(f'  Pre-launch monthly rev:  ${rev_pre:>12,.0f}')
print(f'  Post-launch monthly rev: ${rev_post:>12,.0f}')
print(f'  Incremental monthly rev: ${rev_lift:>12,.0f}  ({rev_lift_pct:.1%})')
print()
print('Recommendation')
print('─' * 60)
print('  The pattern of effects tells a clean story:')
print('  → Conversion lifted AND basket expanded → not just pulling forward intent')
print('  → Bundle mix grew without cannibalizing category → net new behavior')
print('  → Repeat visits increased → feature builds habit, not just transaction')
print()
print('  SCALE. Expand to EMEA in next quarter.')
print('  Priority markets: UK, Germany (highest donor weights in SC model).')
print()
print('Assumptions to pressure-test before global rollout')
print('─' * 60)
print('  • SC donor pool = 9 countries — inference is approximate, not frequentist')
print('  • Secondary outcomes use same weights as primary — assumes parallel donor structure')
print('  • AOV lift may partly reflect product mix shift, not pure basket expansion')
print('  • EMEA conversion baseline is ~30% lower than NA — lift magnitude may differ')
print('  • Recommend user-level holdout in EMEA launch to validate at smaller scale first')

---

## What would change this recommendation

| Condition | Direction | What to do |
|---|---|---|
| AOV lift disappears when controlling for product mix | Basket expansion overstated | Decompose by category before scaling |
| Bundle mix rate flat or negative | Substitution, not expansion | Re-examine feature UX — may be cannibalizing organic browse |
| Repeat visit rate reverts after week 8 | Halo is transient novelty | Wait for 90-day NA data before EMEA commitment |
| EMEA donor pool fails pre-period fit | SC invalid for EMEA | Run holdout test in UK only, use as anchor for EMEA SC |
| Conversion lift concentrated in existing high-intent users | Selection effect | Segment by new vs. returning — feature may not acquire new customers |

---

## Takeaway

Synthetic Control is the right method when you have one treated unit, a structural level difference that makes parallel trends implausible, and enough pre-period data to fit a credible counterfactual. All three applied here.

The secondary effects analysis is where this framework earns its keep. Conversion alone doesn't tell you whether a feature is growing the business or just rearranging it. Basket expansion + bundle growth + repeat visits together tell you: this feature changed behavior, not just captured it.

That distinction is what makes the investment decision defensible — not the p-value.

---
*Python · numpy · pandas · scipy · matplotlib*  
*Synthetic Control methodology following Abadie, Diamond & Hainmueller (2010)*